# PolyWhisper v9 - Per-Language Augmentation

hi/ta/te: no augmentations (they have enough data, augmentations cause degeneration)
bn/mr: full SpecAugment + speed perturb (they benefit dramatically)

Key change: --augment-langs bn,mr flag in train_v3.py

In [ ]:
!pip install -q transformers datasets accelerate peft torch torchaudio evaluate jiwer sentencepiece

In [ ]:
import os, shutil
for d in ['polywhisper_output_gpu0', 'polywhisper_output_gpu1', 'scripts']:
    if os.path.exists(d): shutil.rmtree(d)
os.makedirs('scripts', exist_ok=True)
print('cleaned')

In [ ]:
from huggingface_hub import hf_hub_download
scripts = ['train_v3.py', 'eval_lang_pure.py', 'kaggle_train_resumable.py', 'normalize_ortho.py']
for s in scripts:
    p = hf_hub_download('eulogik/polywhisper', s, repo_type='model')
    shutil.copy(p, f'scripts/{s}')
    print(f'  {s}')

In [ ]:
%%bash
cd scripts
python kaggle_train_resumable.py \
    --base-model /kaggle/input/whisper-small/transformers/default/1/models--openai--whisper-small/snapshots/b9b9f78e5e408756640418859199b237a10ef98c \
    --langs hi,ta,te,bn,mr \
    --model-size small \
    --batch-size 4 \
    --epochs 5 \
    --lr 1e-4 \
    --encoder-lora \
    --save-dir /kaggle/working/polywhisper_output_gpu0 \
    --tag _v9 \
    --max-runtime-hours 11 \
    --wer-eval-every 500 \
    --augment-langs bn,mr

In [ ]:
import os
for lang in ['hi', 'ta', 'te', 'bn', 'mr']:
    result = f'/kaggle/working/polywhisper_output_gpu0/eval_{lang}_pure_fleurs.json'
    if os.path.exists(result):
        import json
        d = json.load(open(result))
        print(f'{lang}: {d.get(chr(119)+chr(101)+chr(114), chr(63)):.1f}%')
    else:
        print(f'{lang}: no eval result')

In [ ]:
from huggingface_hub import HfApi
import os
api = HfApi()
for lang in ['hi', 'ta', 'te', 'bn', 'mr']:
    adapter = f'/kaggle/working/polywhisper_output_gpu0/adapters_v3/{lang}_best_prod.pt'
    if os.path.exists(adapter):
        api.upload_file(path_or_fileobj=adapter, path_in_repo=f'polywhisper_output_gpu0/adapters_v3/{lang}_best_prod.pt', repo_id='eulogik/polywhisper', repo_type='model')
        print(f'  uploaded {lang}')
    eval_result = f'/kaggle/working/polywhisper_output_gpu0/eval_{lang}_pure_fleurs.json'
    if os.path.exists(eval_result):
        api.upload_file(path_or_fileobj=eval_result, path_in_repo=f'polywhisper_output_gpu0/eval_{lang}_pure_fleurs_v9.json', repo_id='eulogik/polywhisper', repo_type='model')
        print(f'  uploaded eval {lang}')